# Construcción del corpus ampliado — dataset de política colombiana

Este notebook toma `dataset_politica_colombiana_original_793.xlsx` (793 filas) y lo amplía con
tres fuentes, de principio a fin y **sin modificar el archivo original**:

| Paso | Fuente | Etiqueta | Cómo se obtiene |
|---|---|---|---|
| 2 | Sitios satíricos (Actualidad Panamericana, El Chigüire Bipolar) | `FALSE` | Sitemaps públicos + clasificador político sobre el texto real |
| 3 | Colombiacheck | `FALSE` | API oficial **Google Fact Check Tools** (datos `ClaimReview`) |
| 4 | El Espectador, sección política | `TRUE` | Sitemap de la sección, solo lo necesario para balancear |

El resultado se guarda en `dataset_politica_colombiana_ampliado.xlsx`, con una hoja `dataset`
(mismas 6 columnas del original) y una hoja `procedencia` que dice de dónde salió cada fila.

## Cómo ejecutarlo

1. **Sube** `dataset_politica_colombiana_original_793.xlsx` al panel de archivos de Colab
   (si no lo haces, el notebook te lo pedirá).
2. **API key:** en el panel 🔑 *Secrets*, crea `FACTCHECK_API_KEY` con tu clave y activa
   "Acceso al notebook". Si no existe, el notebook te la pedirá en un cuadro oculto.
3. Menú **Entorno de ejecución → Ejecutar todo**.

Cada paso de descarga guarda una caché en `cache_corpus/`. Si vuelves a ejecutar todo, no se
repiten las descargas (pon `FORCE_REFRESH = True` para forzarlas). Todo se puede volver a
correr sin duplicar filas: el dataset final se arma desde cero cada vez.

> Los resultados exactos cambian según la fecha de ejecución, porque los sitios siguen publicando.
> El filtro político es por palabras clave: revisa una muestra del resultado antes de entrenar.

## 0. Configuración

In [1]:
# ⚙️ Parámetros — lo único que normalmente querrías cambiar
import os

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE_DIR = "/content" if EN_COLAB else "."
DATASET_ORIGINAL = os.path.join(BASE_DIR, "dataset_politica_colombiana_original_793.xlsx")
DATASET_SALIDA = os.path.join(BASE_DIR, "dataset_politica_colombiana_ampliado.xlsx")
CACHE_DIR = os.path.join(BASE_DIR, "cache_corpus")

FORCE_REFRESH = False        # True = ignora la caché y vuelve a descargar todo

# Colombiacheck: calificaciones que NO se consideran noticia falsa (prefijos, sin tildes)
EXCLUIR_CALIFICACIONES = ("verdadero", "cierto", "correcto", "inchequeable")
MAX_COLOMBIACHECK = None     # None = todos los candidatos; o un número, p. ej. 600

# El Espectador: cuántas noticias reales agregar
BALANCEAR_CLASES = True      # agrega TRUE hasta igualar el número de FALSE
MAX_TRUE_A_AGREGAR = None    # tope opcional (None = sin tope)

# Cortesía con los servidores
WORKERS = 4                  # descargas simultáneas
PAUSA_ENTRE_PETICIONES = 0.3 # segundos por petición y por hilo
USER_AGENT = "Mozilla/5.0 (compatible; dataset-coverage-check/1.0; academic project)"
ROBOTS_AGENT = "dataset-coverage-check"

DESCARGAR_AL_FINAL = True    # descarga el .xlsx final al terminar (solo en Colab)

## 1. Preparación: librerías, utilidades y carga del dataset original

In [2]:
import html
import math
import re
import shutil
import threading
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from urllib import robotparser
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

os.makedirs(CACHE_DIR, exist_ok=True)
COLUMNAS = ["id", "label", "title", "description", "date", "url"]

def conteo(serie) -> dict:
    return {k: int(v) for k, v in serie.value_counts().items()}

# ---------- Texto ----------
def normalize_text(s) -> str:
    s = unicodedata.normalize("NFKD", str(s or "").lower())
    return "".join(c for c in s if not unicodedata.combining(c))

def has_any_phrase(text: str, phrases) -> bool:
    """Coincidencia por palabra/frase completa: 'petro' no coincide con 'petrolero'."""
    return any(re.search(r"\b" + re.escape(p) + r"\b", text) for p in phrases)

# ---------- URLs ----------
def domain_of(url: str) -> str:
    netloc = urlparse(str(url or "")).netloc.lower()
    return netloc[4:] if netloc.startswith("www.") else netloc

def normalize_url(url: str) -> str:
    """Clave para deduplicar: sin esquema, sin www., sin barra final, sin query."""
    p = urlparse(str(url or "").strip())
    return f"{domain_of(url)}{p.path.rstrip('/')}"

# ---------- Fechas ----------
def iso_a_ddmmaaaa(raw: str) -> str:
    try:
        return datetime.fromisoformat(str(raw).strip().replace("Z", "+00:00")).strftime("%d/%m/%Y")
    except Exception:
        return ""

# ---------- HTTP con reintentos ----------
def _crear_sesion():
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",))
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=WORKERS * 2)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers["User-Agent"] = USER_AGENT
    return s

SESION = _crear_sesion()

def fetch_url(url: str, timeout: int = 20) -> str:
    time.sleep(PAUSA_ENTRE_PETICIONES)
    r = SESION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.content.decode("utf-8", errors="replace")

# ---------- robots.txt ----------
_ROBOTS, _ROBOTS_LOCK = {}, threading.Lock()

def permitido_por_robots(url: str) -> bool:
    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    with _ROBOTS_LOCK:
        rp = _ROBOTS.get(base)
        if rp is None:
            rp = robotparser.RobotFileParser()
            try:
                r = SESION.get(base + "/robots.txt", timeout=15)
                if r.status_code in (401, 403):
                    rp.disallow_all = True       # el sitio rechaza incluso robots.txt
                elif r.status_code >= 400:
                    rp.allow_all = True          # sin robots.txt = sin restricciones
                else:
                    rp.parse(r.text.splitlines())
            except Exception:
                rp.allow_all = True
            _ROBOTS[base] = rp
    return rp.can_fetch(ROBOTS_AGENT, url)

# ---------- Concurrencia y caché ----------
def en_paralelo(func, items, etiqueta="procesados", cada=100):
    """func debe capturar sus propias excepciones y devolver un dict."""
    items = list(items)
    salida = []
    if not items:
        return salida
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futuros = [pool.submit(func, x) for x in items]
        for i, fut in enumerate(as_completed(futuros), 1):
            salida.append(fut.result())
            if i % cada == 0 or i == len(items):
                print(f"  ...{i}/{len(items)} {etiqueta}")
    return salida

def con_cache(nombre: str, construir):
    """Carga cache_corpus/<nombre> si existe; si no, lo construye y lo guarda (si no está vacío)."""
    ruta = os.path.join(CACHE_DIR, nombre)
    if os.path.exists(ruta) and not FORCE_REFRESH:
        df = pd.read_csv(ruta, dtype=str, keep_default_na=False)
        print(f"  (usando caché {ruta}: {len(df)} filas — FORCE_REFRESH=True para rehacer)")
        return df
    df = construir()
    if len(df):
        df.to_csv(ruta, index=False, encoding="utf-8")
    return df

# ---------- Carga del dataset original ----------
if not os.path.exists(DATASET_ORIGINAL):
    if EN_COLAB:
        from google.colab import files
        print("No encontré el dataset original. Súbelo ahora:")
        subido = files.upload()
        nombre = next(iter(subido))
        shutil.move(nombre, DATASET_ORIGINAL)
    else:
        raise FileNotFoundError(f"No existe {DATASET_ORIGINAL}")

original = pd.read_excel(DATASET_ORIGINAL, dtype=str).fillna("")
original.columns = [str(c).strip().lower() for c in original.columns]
faltan = set(COLUMNAS) - set(original.columns)
if faltan:
    raise ValueError(f"Faltan columnas en el dataset original: {faltan}")
original = original[COLUMNAS].copy()
original["label"] = original["label"].str.strip().str.upper()

URLS_EXISTENTES = set(original["url"].map(normalize_url))
print(f"Dataset original: {len(original)} filas — {conteo(original['label'])}")

No encontré el dataset original. Súbelo ahora:


Saving dataset_politica_colombiana_original_793.xlsx to dataset_politica_colombiana_original_793.xlsx
Dataset original: 793 filas — {'TRUE': 403, 'FALSE': 390}


## 2. Diagnóstico del corpus original

¿Las filas que ya venían de sitios satíricos están todas etiquetadas como `FALSE`?

In [3]:
SITIOS_SATIRA = {
    "actualidadpanamericana.com": "https://actualidadpanamericana.com/sitemap.xml",
    "elchiguirebipolar.net": "https://www.elchiguirebipolar.net/wp-sitemap.xml",
}

dom_original = original["url"].map(domain_of)
satira_orig = original[dom_original.isin(SITIOS_SATIRA)]
mal = satira_orig[satira_orig["label"] != "FALSE"]

print(f"Filas de sitios satíricos en el original: {len(satira_orig)}")
for dom, n in dom_original[dom_original.isin(SITIOS_SATIRA)].value_counts().items():
    print(f"  {dom:<30} {n}")
if mal.empty:
    print("✔ Todas están etiquetadas como FALSE.")
else:
    print(f"✘ {len(mal)} filas satíricas NO están como FALSE:")
    print(mal[["id", "label", "title", "url"]].to_string(index=False))

dups = original["url"].map(normalize_url).duplicated(keep=False)
print(f"\nURLs repetidas dentro del original (se conservan tal cual): {original[dups]['url'].nunique()}")

Filas de sitios satíricos en el original: 125
  actualidadpanamericana.com     119
  elchiguirebipolar.net          6
✔ Todas están etiquetadas como FALSE.

URLs repetidas dentro del original (se conservan tal cual): 22


## 3. Clasificador político (compartido por todos los pasos)

Señales fuertes (políticos, instituciones, actores armados) bastan por sí solas. Los nombres
de lugares solo cuentan si van acompañados de un término de contexto político. Palabras sueltas
como "gobierno" o "policía" se descartaron porque la sátira las usa como remate en cualquier tema.

In [4]:
STRONG_POLITICAL_SIGNALS = {
    "petro", "gustavo petro", "duque", "ivan duque", "uribe", "alvaro uribe",
    "uribista", "uribismo", "petrista", "petrismo", "santos", "juan manuel santos",
    "samper", "pastrana", "francia marquez", "claudia lopez", "rodolfo hernandez",
    "sergio fajardo", "federico gutierrez", "de la espriella", "abelardo de la espriella",
    "farc", "eln", "esmad", "congreso", "senado", "camara de representantes", "concejo",
    "concejo de bogota", "procuraduria", "contraloria", "defensoria del pueblo",
    "registraduria", "corte constitucional", "corte suprema", "casa de narino",
    "corte penal", "gobernacion", "alcaldia", "alcaldia de bogota", "minhacienda",
    "ministerio de hacienda", "ministerio de defensa", "dnp", "distrito capital",
    "personeria", "veeduria", "curul", "concejal", "diputado", "paro nacional",
    "paro armado", "paro civico", "movilizacion social", "consulta popular",
    "consejo de estado",
}

WEAK_COLOMBIA_SIGNALS = {
    "colombia", "colombiano", "colombiana", "colombianos", "colombianas", "bogota",
    "bogotano", "bogotana", "medellin", "cali", "barranquilla", "cartagena",
    "bucaramanga", "cundinamarca", "antioquia", "valle del cauca", "atlantico",
    "santander", "boyaca", "narino", "cauca", "cordoba", "tolima", "huila", "dane",
}

POLITICAL_CONTEXT_TERMS = {
    "gobierno colombiano", "gobierno nacional", "gobierno distrital", "gobierno de bogota",
    "policia nacional", "presidente", "presidenta", "presidencia", "ministro", "ministra",
    "ministerio", "senador", "senadora", "representante a la camara", "alcalde",
    "alcaldesa", "gobernador", "gobernadora", "defensoria", "eleccion", "elecciones",
    "campana electoral", "candidato", "candidata", "reforma tributaria",
    "reforma pensional", "reforma laboral", "reforma a la salud", "decreto",
    "proyecto de ley", "impuesto", "impuestos", "paz total", "acuerdo de paz",
    "guerrilla", "paramilitar", "ejercito", "partido politico", "oposicion", "coalicion",
    "bancada", "mocion de censura", "revocatoria", "plebiscito", "referendo",
    "corrupcion", "escandalo politico", "secretaria de educacion",
    "secretaria de movilidad", "secretaria de salud", "secretaria de gobierno",
    "ordenamiento territorial", "licitacion", "contratacion publica", "contratista",
    "contratistas", "obra publica", "paro de transporte", "marcha de protesta",
    "protesta social", "acto legislativo", "salario minimo", "iva",
}

def clasificar(*textos, exigir_lugar=True):
    """Devuelve (es_politica, razon). exigir_lugar=False para fuentes que ya son colombianas."""
    t = normalize_text(" . ".join(str(x or "") for x in textos))
    if has_any_phrase(t, STRONG_POLITICAL_SIGNALS):
        return True, "senal_politica_fuerte"
    contexto = has_any_phrase(t, POLITICAL_CONTEXT_TERMS)
    if not exigir_lugar and contexto:
        return True, "contexto_politico"
    lugar = has_any_phrase(t, WEAK_COLOMBIA_SIGNALS)
    if lugar and contexto:
        return True, "lugar+contexto_politico"
    return False, "solo_lugar" if lugar else "sin_senal_politica"

# Pruebas rápidas
assert clasificar("Petro anuncia algo")[0]
assert not clasificar("Nuevo pozo petrolero en California")[0]
assert not clasificar("Gol de Millonarios en Bogotá")[0]
assert clasificar("Alcalde de Medellín anuncia decreto")[0]
print("Clasificador listo.")

Clasificador listo.


## 4. Sátira política colombiana → `FALSE`

Recorre los sitemaps de los dos sitios satíricos, filtra por el slug (barato), descarta páginas
que no son artículos (etiquetas, categorías, autores) y descarga cada candidato **una sola vez**
para clasificarlo y, en la misma petición, sacar título limpio, descripción del cuerpo y fecha.

Esto reemplaza los antiguos pasos 2, 3 y 3b: el error del encabezado de autor con nombres de dos
palabras y la página de etiqueta `/etiqueta/juan-manuel-santos/` ya quedan resueltos en el código,
sin correcciones a mano por id.

In [5]:
SLUG_TOKENS = {
    "colombia", "colombiano", "colombiana", "colombianos", "colombianas", "bogota",
    "bogotano", "bogotana", "medellin", "cali", "barranquilla", "cartagena", "bucaramanga",
    "cundinamarca", "antioquia", "atlantico", "santander", "boyaca", "narino", "cauca",
    "cordoba", "tolima", "huila", "petro", "duque", "uribe", "uribista", "uribismo",
    "santos", "samper", "pastrana", "gaitan", "farc", "eln", "dane", "dnp", "minhacienda",
    "senado", "registraduria", "contraloria", "procuraduria", "defensoria", "esmad",
    "transmilenio", "petrista", "petrismo", "espriella",
}
NO_ARTICULO_RE = re.compile(r"/(etiqueta|tag|categoria|category|autor|author|page|feed|wp-content)/", re.I)
SITEMAP_EXCLUIR_RE = re.compile(r"image|taxonom|users|author", re.I)

TITLE_SUFFIX_RE = re.compile(r"\s*[-|–]\s*(Actualidad Panamericana|El Chig[üu]ire Bipolar)\s*$", re.I)
MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
AP_FECHA_RE = re.compile(
    r"Publicado el\s+(\d{1,2})\s*(?:de\s+)?([A-Za-zñÁÉÍÓÚáéíóú]+)\s*,?\s*(?:de\s+)?(\d{4})", re.I)
# Encabezado "por <autor de 1 a 5 palabras> en <categoría> ." justo después de la fecha
AP_BYLINE_RE = re.compile(r"^\s*por\s+(?:\S+\s+){1,5}?en\s+[^.]{0,80}\.\s*", re.I)
CHIGUIRE_FECHA_URL_RE = re.compile(r"/(\d{2})-(\d{2})-(\d{4})/")


def _locs(xml: str):
    return [html.unescape(u.strip()) for u in re.findall(r"<loc>([^<]+)</loc>", xml)]

def urls_de_sitemap(indice: str):
    urls, pendientes, vistos = [], [indice], set()
    while pendientes:
        sm = pendientes.pop()
        if sm in vistos:
            continue
        vistos.add(sm)
        try:
            xml = fetch_url(sm)
        except Exception as e:
            print(f"    !! no se pudo leer {sm}: {e}")
            continue
        if "<sitemapindex" in xml:
            pendientes.extend(u for u in _locs(xml) if not SITEMAP_EXCLUIR_RE.search(u))
        else:
            urls.extend(_locs(xml))
    return urls

def slug_candidato(url: str) -> bool:
    tokens = re.split(r"[^a-z0-9]+", normalize_text(urlparse(url).path))
    return any(t in SLUG_TOKENS for t in tokens)

def _meta(soup, **attrs):
    tag = soup.find("meta", attrs=attrs)
    return (tag.get("content") or "").strip() if tag else ""

def _recortar(texto: str, n=500) -> str:
    texto = re.sub(r"\s+", " ", texto).strip()
    if len(texto) <= n:
        return texto
    corte = texto[:n]
    p = corte.rfind(". ")
    return corte[: p + 1] if p > 80 else corte

def procesar_satira(url: str) -> dict:
    try:
        soup = BeautifulSoup(fetch_url(url), "html.parser")
    except Exception as e:
        return {"url": url, "error": str(e)[:200]}

    titulo = _meta(soup, property="og:title") or (soup.title.get_text(strip=True) if soup.title else "")
    titulo = TITLE_SUFFIX_RE.sub("", titulo).strip()
    meta_desc = _meta(soup, property="og:description") or _meta(soup, name="description")
    fecha = iso_a_ddmmaaaa(_meta(soup, property="article:published_time"))
    parrafos = [p.get_text(" ", strip=True) for p in
                (soup.select("div.entry-content p") or soup.select("article p"))]
    parrafos = [p for p in parrafos if p]

    if domain_of(url) == "actualidadpanamericana.com":
        cuerpo = " ".join(parrafos)
        m = AP_FECHA_RE.search(cuerpo)
        if m:
            d, mes, a = m.groups()
            if MESES.get(mes.lower()):
                fecha = f"{int(d):02d}/{MESES[mes.lower()]:02d}/{a}"
            cuerpo = AP_FECHA_RE.sub("", cuerpo, count=1)
            cuerpo = AP_BYLINE_RE.sub("", cuerpo, count=1)
        descripcion = _recortar(cuerpo)
        if len(descripcion) < 40:
            descripcion = meta_desc
    else:  # elchiguirebipolar.net
        m = CHIGUIRE_FECHA_URL_RE.search(url)
        if m:
            fecha = f"{m.group(1)}/{m.group(2)}/{m.group(3)}"
        descripcion = meta_desc
        if len(descripcion) < 40:
            largos = [p for p in parrafos if len(p) > 40]
            descripcion = _recortar(largos[0]) if largos else descripcion

    es_pol, razon = clasificar(titulo, meta_desc, " ".join(parrafos[:6]), exigir_lugar=True)
    return {"url": url, "title": titulo, "description": descripcion or titulo, "date": fecha,
            "es_politica": str(es_pol), "razon": razon, "error": ""}


def construir_satira():
    candidatos = []
    for dom, indice in SITIOS_SATIRA.items():
        print(f"Sitemap de {dom} ...")
        if not permitido_por_robots(indice):
            print("  robots.txt no permite este sitio; se omite.")
            continue
        todas = urls_de_sitemap(indice)
        articulos = [u for u in todas if domain_of(u) == dom and not NO_ARTICULO_RE.search(u)]
        por_slug = [u for u in articulos if slug_candidato(u)]
        nuevas = [u for u in dict.fromkeys(por_slug) if normalize_url(u) not in URLS_EXISTENTES]
        nuevas = [u for u in nuevas if permitido_por_robots(u)]
        print(f"  {len(todas)} URLs en el sitemap, {len(articulos)} artículos, "
              f"{len(por_slug)} pasan el filtro de slug, {len(nuevas)} nuevas")
        candidatos.extend(nuevas)
    print(f"\nDescargando y clasificando {len(candidatos)} artículos...")
    return pd.DataFrame(en_paralelo(procesar_satira, candidatos, "artículos"),
                        columns=["url", "title", "description", "date", "es_politica", "razon", "error"])


satira_raw = con_cache("satira_clasificada.csv", construir_satira).fillna("")
fallos = satira_raw[satira_raw["error"] != ""]
satira = satira_raw[(satira_raw["error"] == "") & (satira_raw["es_politica"] == "True")].copy()
satira = satira[~satira["url"].map(normalize_url).isin(URLS_EXISTENTES)]
satira = satira.loc[~satira["url"].map(normalize_url).duplicated()]

print(f"\nClasificados: {len(satira_raw) - len(fallos)}  |  fallos de descarga: {len(fallos)}")
print(f"Razones: {conteo(satira_raw['razon'])}")
print(f"✔ Sátira política nueva → FALSE: {len(satira)} filas")
con_byline = satira["description"].str.match(r"(?i)\s*por\s").sum()
print(f"  sin fecha: {(satira['date'] == '').sum()}  |  descripción que empieza con 'por …': {con_byline}")
display(satira[["title", "date", "url"]].head(8))

Sitemap de actualidadpanamericana.com ...
  2621 URLs en el sitemap, 2621 artículos, 752 pasan el filtro de slug, 694 nuevas
Sitemap de elchiguirebipolar.net ...
  11528 URLs en el sitemap, 11528 artículos, 158 pasan el filtro de slug, 152 nuevas

Descargando y clasificando 846 artículos...
  ...100/846 artículos
  ...200/846 artículos
  ...300/846 artículos
  ...400/846 artículos
  ...500/846 artículos
  ...600/846 artículos
  ...700/846 artículos
  ...800/846 artículos
  ...846/846 artículos

Clasificados: 846  |  fallos de descarga: 0
Razones: {'senal_politica_fuerte': 505, 'solo_lugar': 216, 'lugar+contexto_politico': 121, 'sin_senal_politica': 4}
✔ Sátira política nueva → FALSE: 626 filas
  sin fecha: 0  |  descripción que empieza con 'por …': 7


,title,date,url
0,Mascota de expresidente Samper sería responsab...,24/04/2017,https://actualidadpanamericana.com/mascota-exp...
1,Expulsan a pasajero de ruta sobrevendida de Tr...,25/04/2017,https://actualidadpanamericana.com/expulsan-pa...
2,ESMAD lanzó gas lacrimógeno a personas que ya ...,20/04/2017,https://actualidadpanamericana.com/esmad-lanzo...
3,Colombia ya es autosuficiente y pronto comenza...,21/04/2017,https://actualidadpanamericana.com/colombia-ya...
5,"""Por algo que le mandaron por el whatsapp"", Pe...",07/05/2017,https://actualidadpanamericana.com/algo-leyo-i...
6,Intérpretes de lenguaje de señas piden medio t...,06/05/2017,https://actualidadpanamericana.com/traductores...
7,Trump olvidó visita de Santos y se fue de viaje,17/05/2017,https://actualidadpanamericana.com/trump-olvid...
8,"Santos le sigue saliendo a Uribe en ""Personas ...",23/05/2017,https://actualidadpanamericana.com/santos-le-s...


## 5. Colombiacheck → `FALSE` (vía Google Fact Check Tools API)

Colombiacheck bloquea las IP de datacenter como las de Colab, así que en vez de rasparlo se usa
la API oficial de Google, que expone los mismos datos `ClaimReview` que el sitio publica.
Se excluyen las calificaciones de `EXCLUIR_CALIFICACIONES`; el resto sigue la convención del
corpus original (afirmación revisada por un verificador = `FALSE`).

In [6]:
FACTCHECK_URL = "https://factchecktools.googleapis.com/v1alpha1/claims:search"

def obtener_api_key() -> str:
    key = os.environ.get("FACTCHECK_API_KEY", "").strip()
    if not key and EN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get("FACTCHECK_API_KEY") or "").strip()
        except Exception as e:
            print(f"  No pude leer el secreto FACTCHECK_API_KEY ({type(e).__name__}).")
    if not key:
        from getpass import getpass
        key = getpass("Pega tu API key de Fact Check Tools (no se mostrará): ").strip()
    return key

def construir_colombiacheck(max_paginas=300):
    key = obtener_api_key()
    filas, token = [], None
    for pagina in range(max_paginas):
        params = {"reviewPublisherSiteFilter": "colombiacheck.com", "pageSize": 100, "key": key}
        if token:
            params["pageToken"] = token
        r = requests.get(FACTCHECK_URL, params=params, timeout=30)
        if r.status_code != 200:
            msg = f"API HTTP {r.status_code}: {r.text[:300]}"
            if pagina == 0:
                raise RuntimeError(msg + "\n→ Revisa la API key y que la API esté habilitada en tu proyecto.")
            print(f"  !! {msg} (se conservan las {len(filas)} filas descargadas)")
            break
        data = r.json()
        for c in data.get("claims", []):
            texto = (c.get("text") or "").strip()
            for rev in c.get("claimReview", []):
                url = rev.get("url", "")
                if texto and "colombiacheck.com" in url:
                    filas.append({
                        "title": texto,
                        "description": (rev.get("title") or "").strip(),
                        "date": iso_a_ddmmaaaa(rev.get("reviewDate") or c.get("claimDate") or ""),
                        "calificacion": (rev.get("textualRating") or "").strip(),
                        "url": url,
                    })
        if pagina % 5 == 0:
            print(f"  página {pagina}: {len(filas)} chequeos")
        token = data.get("nextPageToken")
        if not token:
            break
        time.sleep(0.3)
    print(f"  total descargado: {len(filas)} chequeos")
    return pd.DataFrame(filas, columns=["title", "description", "date", "calificacion", "url"])


cc_raw = con_cache("colombiacheck_api.csv", construir_colombiacheck).fillna("")

print("\nDistribución de calificaciones:")
for cal, n in cc_raw["calificacion"].value_counts().items():
    marca = "  (excluida)" if normalize_text(cal).startswith(EXCLUIR_CALIFICACIONES) else ""
    print(f"  {cal or '(vacía)':<30} {n}{marca}")

cc = cc_raw[~cc_raw["calificacion"].map(normalize_text).str.startswith(EXCLUIR_CALIFICACIONES)].copy()
cc["politica"] = [clasificar(t, d, exigir_lugar=False)[0] for t, d in zip(cc["title"], cc["description"])]
cc = cc[cc["politica"]].drop(columns="politica")
claves_previas = URLS_EXISTENTES | set(satira["url"].map(normalize_url))
cc = cc[~cc["url"].map(normalize_url).isin(claves_previas)]
cc = cc.loc[~cc["url"].map(normalize_url).duplicated()]
if MAX_COLOMBIACHECK:
    cc = cc.head(MAX_COLOMBIACHECK)

print(f"\n✔ Colombiacheck nuevo → FALSE: {len(cc)} filas")
display(cc[["calificacion", "title", "date"]].head(8))

  página 0: 100 chequeos
  página 5: 600 chequeos
  página 10: 1100 chequeos
  página 15: 1601 chequeos
  página 20: 2101 chequeos
  página 25: 2603 chequeos
  página 30: 3015 chequeos
  total descargado: 3015 chequeos

Distribución de calificaciones:
  Falso                          2301
  Cuestionable                   607
  Verdadero pero...              54  (excluida)
  Verdadero                      52  (excluida)
  Duque acierta en cifras de extradiciones a EE.UU. y de laboratorios destruidos, pero se equivoca en las de erradicación 1

✔ Colombiacheck nuevo → FALSE: 1336 filas


,calificacion,title,date
0,Falso,Video capturados por paro armado en Santa Mart...,30/09/2026
1,Falso,"Video de alias ""Calarcá"" hablando contra Petro...",29/09/2026
3,Cuestionable,Video de reciente atentado ocurrido durante el...,24/09/2026
4,Cuestionable,Video reciente de encapuchados en la Universid...,23/09/2026
5,Falso,Video de la casa que Petro compró por un milló...,22/09/2026
8,Cuestionable,Victoria del Tigre: recuperó 9 menores y 41 di...,18/09/2026
9,Falso,Pinocho advierte a De la Espriella que el país...,17/09/2026
10,Falso,"La MOE UE confirmó que sí hubo ""voto fusil"" en...",16/09/2026


## 6. Noticias reales de El Espectador → `TRUE` (balanceo)

Se calcula cuántas filas `TRUE` hacen falta para igualar las `FALSE` y se descargan solo esas,
muestreando páginas repartidas por todo el archivo de la sección política para tener diversidad
temporal. Las URLs de paginación del sitemap vienen con `&amp;`, por eso se des-escapan.

In [7]:
ESPECTADOR_INDICE = ("https://www.elespectador.com/arc/outboundfeeds/sitemap-index/"
                     "section/politica/?outputType=xml")
ESPECTADOR_ENTRADA_RE = re.compile(r"<loc>([^<]+)</loc>\s*<lastmod>([^<]+)</lastmod>", re.S)
ESPECTADOR_SUFIJO_RE = re.compile(r"\s*[|\-–]\s*El Espectador\s*$", re.I)

n_false = (original["label"] == "FALSE").sum() + len(satira) + len(cc)
n_true = (original["label"] == "TRUE").sum()
necesarias = max(0, n_false - n_true) if BALANCEAR_CLASES else 0
if MAX_TRUE_A_AGREGAR is not None:
    necesarias = min(necesarias, MAX_TRUE_A_AGREGAR)
n_false, n_true = int(n_false), int(n_true)
print(f"FALSE tras pasos 4-5: {n_false}  |  TRUE: {n_true}  →  hacen falta {necesarias} TRUE")

def procesar_espectador(entrada) -> dict:
    url, lastmod = entrada
    try:
        soup = BeautifulSoup(fetch_url(url), "html.parser")
    except Exception as e:
        return {"url": url, "error": str(e)[:200]}
    titulo = _meta(soup, property="og:title") or (soup.title.get_text(strip=True) if soup.title else "")
    titulo = ESPECTADOR_SUFIJO_RE.sub("", titulo).strip()
    desc = _meta(soup, property="og:description") or _meta(soup, name="description")
    fecha = iso_a_ddmmaaaa(_meta(soup, property="article:published_time")) or iso_a_ddmmaaaa(lastmod)
    ok = bool(titulo) and len(desc) > 20
    return {"url": url, "title": titulo, "description": desc, "date": fecha,
            "error": "" if ok else "sin_titulo_o_descripcion"}

def construir_espectador():
    if necesarias == 0:
        return pd.DataFrame(columns=["url", "title", "description", "date", "error"])
    if not permitido_por_robots(ESPECTADOR_INDICE):
        print("robots.txt no permite el sitemap de El Espectador; se omite.")
        return pd.DataFrame(columns=["url", "title", "description", "date", "error"])

    subpaginas = [html.unescape(u) for u in re.findall(r"<loc>([^<]+)</loc>", fetch_url(ESPECTADOR_INDICE))]
    n_pag = min(len(subpaginas), math.ceil(necesarias * 1.6 / 100) + 2)
    idx = sorted({round(i * (len(subpaginas) - 1) / max(1, n_pag - 1)) for i in range(n_pag)})
    print(f"  {len(subpaginas)} páginas de sitemap; muestreando {len(idx)} repartidas en el archivo")

    ya = URLS_EXISTENTES | set(satira["url"].map(normalize_url)) | set(cc["url"].map(normalize_url))
    entradas = {}
    for i in idx:
        try:
            for loc, lastmod in ESPECTADOR_ENTRADA_RE.findall(fetch_url(subpaginas[i])):
                loc = html.unescape(loc.strip())
                clave = normalize_url(loc)
                if "/politica/" in loc and clave not in ya and permitido_por_robots(loc):
                    entradas.setdefault(clave, (loc, lastmod.strip()))
        except Exception as e:
            print(f"    !! página {i} falló: {e}")
    entradas = list(entradas.values())
    print(f"  {len(entradas)} URLs candidatas nuevas")

    # Descarga por lotes hasta tener suficientes artículos válidos
    resultados, lote = [], 200
    for ini in range(0, len(entradas), lote):
        resultados += en_paralelo(procesar_espectador, entradas[ini:ini + lote], "artículos", cada=lote)
        validos = sum(1 for r in resultados if not r.get("error"))
        print(f"  → {validos}/{necesarias} artículos válidos")
        if validos >= necesarias:
            break
    return pd.DataFrame(resultados, columns=["url", "title", "description", "date", "error"])


esp_raw = con_cache("elespectador_articulos.csv", construir_espectador).fillna("")
esp = esp_raw[esp_raw["error"] == ""].copy()
esp = esp.loc[~esp["url"].map(normalize_url).duplicated()].head(necesarias)
if len(esp) < necesarias:
    print(f"⚠ Solo hay {len(esp)} de {necesarias}. Si cambiaste parámetros, usa FORCE_REFRESH=True.")
print(f"✔ El Espectador nuevo → TRUE: {len(esp)} filas")
display(esp[["title", "date"]].head(8))

FALSE tras pasos 4-5: 2352  |  TRUE: 403  →  hacen falta 1949 TRUE
  101 páginas de sitemap; muestreando 34 repartidas en el archivo
  3275 URLs candidatas nuevas
  ...200/200 artículos
  → 200/1949 artículos válidos
  ...200/200 artículos
  → 400/1949 artículos válidos
  ...200/200 artículos
  → 600/1949 artículos válidos
  ...200/200 artículos
  → 800/1949 artículos válidos
  ...200/200 artículos
  → 1000/1949 artículos válidos
  ...200/200 artículos
  → 1200/1949 artículos válidos
  ...200/200 artículos
  → 1400/1949 artículos válidos
  ...200/200 artículos
  → 1600/1949 artículos válidos
  ...200/200 artículos
  → 1800/1949 artículos válidos
  ...200/200 artículos
  → 2000/1949 artículos válidos
✔ El Espectador nuevo → TRUE: 1949 filas


,title,date
0,Registraduría despliega estrategia de identifi...,05/10/2026
1,Abelardo de la Espriella envió mensaje de apoy...,05/10/2026
2,De la Espriella anunció control especial en Si...,05/10/2026
3,Cancillería señala presuntas irregularidades e...,05/10/2026
4,Reforma Rural Integral enfrenta recorte presup...,05/10/2026
5,¿Por qué De la Espriella plantea regular prote...,04/10/2026
6,"""Para ganar la guerra hay que ir a por el 2027...",04/10/2026
7,Iván Cepeda: “No nos vamos a dejar vetar ni ce...,04/10/2026


## 7. Ensamblar y guardar el dataset ampliado

In [8]:
nuevas = pd.concat([
    satira.assign(label="FALSE", origen="satira", calificacion=""),
    cc.assign(label="FALSE", origen="colombiacheck"),
    esp.assign(label="TRUE", origen="elespectador", calificacion=""),
], ignore_index=True)

claves = nuevas["url"].map(normalize_url)
nuevas = nuevas[~claves.isin(URLS_EXISTENTES) & ~claves.duplicated()].copy()
for col in ("title", "description"):
    nuevas[col] = nuevas[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

siguiente_id = pd.to_numeric(original["id"], errors="coerce").max()
siguiente_id = int(siguiente_id) + 1 if pd.notna(siguiente_id) else 1
nuevas["id"] = [str(i) for i in range(siguiente_id, siguiente_id + len(nuevas))]

final = pd.concat([original[COLUMNAS], nuevas[COLUMNAS]], ignore_index=True)
procedencia = pd.concat([
    original[["id", "url"]].assign(origen="original", calificacion=""),
    nuevas[["id", "url", "origen", "calificacion"]],
], ignore_index=True)

with pd.ExcelWriter(DATASET_SALIDA, engine="openpyxl") as w:
    final.to_excel(w, sheet_name="dataset", index=False)
    procedencia.to_excel(w, sheet_name="procedencia", index=False)

print(f"Guardado: {DATASET_SALIDA}")
print(f"  original: {len(original)}  +  nuevas: {len(nuevas)}  =  {len(final)} filas")
print(f"  nuevas por origen: {conteo(nuevas['origen'])}")

Guardado: /content/dataset_politica_colombiana_ampliado.xlsx
  original: 793  +  nuevas: 3911  =  4704 filas
  nuevas por origen: {'elespectador': 1949, 'colombiacheck': 1336, 'satira': 626}


## 8. Verificación final

In [9]:
v = pd.read_excel(DATASET_SALIDA, sheet_name="dataset", dtype=str).fillna("")
p = pd.read_excel(DATASET_SALIDA, sheet_name="procedencia", dtype=str).fillna("")

print("Filas totales:", len(v))
print("Etiquetas:", conteo(v["label"]))
print("Por origen:", conteo(p["origen"]))
print("IDs repetidos:", v["id"].duplicated().sum())
print("Títulos vacíos:", (v["title"].str.len() == 0).sum())
print("Descripciones vacías o < 10 caracteres:", (v["description"].str.len() < 10).sum())
print("Fechas vacías:", (v["date"] == "").sum())

claves = v["url"].map(normalize_url)
nuevas_mask = p["origen"] != "original"
print("URLs repetidas entre filas nuevas o nuevas vs. original:",
      (claves[nuevas_mask].isin(set(claves[~nuevas_mask])) | claves[nuevas_mask].duplicated()).sum())

print("\nMuestra aleatoria por origen (revísala antes de entrenar):")
muestra = v.merge(p[["id", "origen"]], on="id")
for origen, grupo in muestra.groupby("origen"):
    print(f"\n--- {origen} ---")
    for _, r in grupo.sample(min(3, len(grupo)), random_state=0).iterrows():
        print(f"  [{r['label']}] {r['title'][:90]}")

Filas totales: 4704
Etiquetas: {'FALSE': 2352, 'TRUE': 2352}
Por origen: {'elespectador': 1949, 'colombiacheck': 1336, 'original': 793, 'satira': 626}
IDs repetidos: 0
Títulos vacíos: 0
Descripciones vacías o < 10 caracteres: 0
Fechas vacías: 0
URLs repetidas entre filas nuevas o nuevas vs. original: 0

Muestra aleatoria por origen (revísala antes de entrenar):

--- colombiacheck ---
  [FALSE] Ejercitó repartió donaciones en Cabecera, barrio rico de Bucaramanga
  [FALSE] Nicolás Petro dijo en Semana que su padre consumía drogas y lo maltrataba
  [FALSE] Mujer junto al sicario es edilesa del Centro Democrático

--- elespectador ---
  [TRUE] Esta es la ruta que le falta al presupuesto de 2027 para aprobarse: le quedan 18 días
  [TRUE] Cancillería alista el Foro Mundial de Migración: estas son las claves de la cumbre
  [TRUE] Estos son los detalles de la cumbre opositora entre uribistas y Cambio Radical

--- original ---
  [TRUE] Partido Liberal: Gaviria decide irse con Federico Gutiérrez

In [10]:
if DESCARGAR_AL_FINAL and EN_COLAB:
    from google.colab import files
    files.download(DATASET_SALIDA)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>